# Profilage DVF 2022 — Etape 2d : anomalies de completude

Les etapes precedentes ont decouvert la structure du fichier (etape 1), visualise les constats principaux (etape 2a), explore les lignes indistinguables (etape 2b) et examine les distributions (etape 2c).

Ce notebook approfondit trois anomalies de completude identifiees lors de la revue du document de profilage :

1. Coherence entre numeros de lots et Surfaces Carrez
2. Profil des lignes sans adresse (Voie, Code postal, Code voie)
3. Profil des lignes sans valeur fonciere

## Mode d'emploi

1. Le chemin est défini dans la **cellule 2**.
2. Executer les cellules dans l'ordre (Maj + Entree).

## Cellule 1 — Installation

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "duckdb", "-q"])
print("Bibliotheques pretes.")

Bibliotheques pretes.


## Cellule 2 — Reglages

**Seule cellule a modifier.**

In [2]:
import duckdb
import os
from pathlib import Path

# Chemin a renseigner
FICHIER = Path(r"./data/dvf-2022.parquet")
SORTIE = Path(r"./figures")
SORTIE.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
pq = str(FICHIER)

assert FICHIER.exists(), f"Fichier introuvable : {FICHIER}"
nb_lignes = con.execute(f"SELECT count(*) FROM '{pq}'").fetchone()[0]
print(f"Fichier : {FICHIER.name}")
print(f"Lignes : {nb_lignes:,}".replace(",", " "))

Fichier : dvf-2022.parquet
Lignes : 4 617 590


## Cellule 3 — Coherence lots / Surfaces Carrez

Pour chaque paire (numero de lot, Surface Carrez du lot), combien de lignes ont le lot sans la Carrez, la Carrez sans le lot, ou les deux ? Une Surface Carrez renseignee sans numero de lot serait une anomalie.

In [3]:
lots = [
    ("1er lot",  "Surface Carrez du 1er lot"),
    ("2eme lot", "Surface Carrez du 2eme lot"),
    ("3eme lot", "Surface Carrez du 3eme lot"),
    ("4eme lot", "Surface Carrez du 4eme lot"),
    ("5eme lot", "Surface Carrez du 5eme lot"),
]

print(f"{'Paire':<40} {'Lot sans Carrez':>16} {'Carrez sans lot':>16} {'Les deux':>12}")
print("-" * 88)

for lot_col, carrez_col in lots:
    r = con.execute(f"""
        SELECT
            COUNT(CASE WHEN \"{lot_col}\" IS NOT NULL AND \"{carrez_col}\" IS NULL THEN 1 END),
            COUNT(CASE WHEN \"{lot_col}\" IS NULL AND \"{carrez_col}\" IS NOT NULL THEN 1 END),
            COUNT(CASE WHEN \"{lot_col}\" IS NOT NULL AND \"{carrez_col}\" IS NOT NULL THEN 1 END)
        FROM '{pq}'
    """).fetchone()
    label = f"{lot_col} / {carrez_col}"
    print(f"  {label:<38} {r[0]:>14,} {r[1]:>14,} {r[2]:>10,}".replace(",", " "))

Paire                                     Lot sans Carrez  Carrez sans lot     Les deux
----------------------------------------------------------------------------------------
  1er lot / Surface Carrez du 1er lot         1 078 513              0    419 277
  2eme lot / Surface Carrez du 2eme lot         316 364              0    141 642
  3eme lot / Surface Carrez du 3eme lot          66 408              0     15 990
  4eme lot / Surface Carrez du 4eme lot          22 536              0      3 908
  5eme lot / Surface Carrez du 5eme lot           9 889              0      1 452


## Cellule 4 — Taux de remplissage lots et Surfaces Carrez

Vue d'ensemble : pour chaque colonne de lot et de Surface Carrez, le nombre de valeurs non nulles et le pourcentage de remplissage. Le taux de remplissage de la Surface Carrez devrait etre inferieur ou egal a celui du numero de lot correspondant.

In [4]:
colonnes = [
    "1er lot", "Surface Carrez du 1er lot",
    "2eme lot", "Surface Carrez du 2eme lot",
    "3eme lot", "Surface Carrez du 3eme lot",
    "4eme lot", "Surface Carrez du 4eme lot",
    "5eme lot", "Surface Carrez du 5eme lot",
]

print(f"{'Colonne':<35} {'Non null':>12} {'% rempli':>10} {'% manquant':>12}")
print("-" * 72)

for col in colonnes:
    r = con.execute(f"""
        SELECT
            COUNT(\"{col}\"),
            ROUND(100.0 * COUNT(\"{col}\") / COUNT(*), 2),
            ROUND(100.0 * (COUNT(*) - COUNT(\"{col}\")) / COUNT(*), 2)
        FROM '{pq}'
    """).fetchone()
    print(f"  {col:<33} {r[0]:>10,} {r[1]:>9.2f} % {r[2]:>10.2f} %".replace(",", " "))

Colonne                                 Non null   % rempli   % manquant
------------------------------------------------------------------------
  1er lot                            1 497 790     32.44 %      67.56 %
  Surface Carrez du 1er lot            419 277      9.08 %      90.92 %
  2eme lot                             458 006      9.92 %      90.08 %
  Surface Carrez du 2eme lot           141 642      3.07 %      96.93 %
  3eme lot                              82 398      1.78 %      98.22 %
  Surface Carrez du 3eme lot            15 990      0.35 %      99.65 %
  4eme lot                              26 444      0.57 %      99.43 %
  Surface Carrez du 4eme lot             3 908      0.08 %      99.92 %
  5eme lot                              11 341      0.25 %      99.75 %
  Surface Carrez du 5eme lot             1 452      0.03 %      99.97 %


## Cellule 5 — Lignes sans adresse

Les colonnes Voie, Code postal et Code voie ont un manquant de 0,83 a 0,84 %. Quel est le profil de ces lignes ? Type de bien, nature de mutation, departements les plus concernes.

In [5]:
nb_sans_cp = con.execute(f"SELECT COUNT(*) FROM '{pq}' WHERE \"Code postal\" IS NULL").fetchone()[0]
print(f"Lignes sans Code postal : {nb_sans_cp:,} ({100.0 * nb_sans_cp / nb_lignes:.2f} %)".replace(",", " "))
print()

# Par type de local
print("Par type de local :")
r = con.execute(f"""
    SELECT
        COALESCE(\"Type local\", '(vide)') AS type_local,
        COUNT(*) AS nb,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM '{pq}'
    WHERE \"Code postal\" IS NULL
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchall()
for typ, nb, pct in r:
    print(f"  {typ:<45} {nb:>8,} ({pct:>5.1f} %)".replace(",", " "))

print()
print("Par nature de mutation :")
r2 = con.execute(f"""
    SELECT
        \"Nature mutation\",
        COUNT(*) AS nb,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM '{pq}'
    WHERE \"Code postal\" IS NULL
    GROUP BY 1
    ORDER BY 2 DESC
""").fetchall()
for nat, nb, pct in r2:
    print(f"  {nat:<45} {nb:>8,} ({pct:>5.1f} %)".replace(",", " "))

print()
print("Top 10 departements :")
r3 = con.execute(f"""
    SELECT \"Code departement\", COUNT(*) AS nb
    FROM '{pq}'
    WHERE \"Code postal\" IS NULL
    GROUP BY 1 ORDER BY 2 DESC LIMIT 10
""").fetchall()
for dept, nb in r3:
    print(f"  Departement {dept:<6} : {nb:>6,} lignes".replace(",", " "))

Lignes sans Code postal : 38 707 (0.84 %)

Par type de local :
  (vide)                                          38 632 ( 99.8 %)
  Dépendance                                          38 (  0.1 %)
  Maison                                              21 (  0.1 %)
  Appartement                                         14 (  0.0 %)
  Local industriel. commercial ou assimilé             2 (  0.0 %)

Par nature de mutation :
  Vente en l'état futur d'achèvement              36 709 ( 94.8 %)
  Vente                                            1 797 (  4.6 %)
  Expropriation                                      105 (  0.3 %)
  Vente terrain à bâtir                               57 (  0.1 %)
  Echange                                             37 (  0.1 %)
  Adjudication                                         2 (  0.0 %)

Top 10 departements :
  Departement 74     :  3 520 lignes
  Departement 92     :  2 240 lignes
  Departement 34     :  1 809 lignes
  Departement 33     :  1 643 lignes
  D

## Cellule 6 — Lignes sans valeur fonciere : profil

31 142 lignes (0,67 %) n'ont pas de valeur fonciere. Ce manquant est-il lie au mecanisme multi-lignes (parcelles dont le prix est porte par une autre ligne de la meme mutation) ?

In [6]:
nb_sans_vf = con.execute(f"SELECT COUNT(*) FROM '{pq}' WHERE \"Valeur fonciere\" IS NULL").fetchone()[0]
print(f"Lignes sans valeur fonciere : {nb_sans_vf:,} ({100.0 * nb_sans_vf / nb_lignes:.2f} %)".replace(",", " "))
print()

# Par type de local et nature de mutation
print("Par type de local et nature de mutation (top 10) :")
r = con.execute(f"""
    SELECT
        COALESCE(\"Type local\", '(vide)') AS type_local,
        \"Nature mutation\",
        COUNT(*) AS nb,
        ROUND(100.0 * COUNT(*) / {nb_sans_vf}, 1) AS pct
    FROM '{pq}'
    WHERE \"Valeur fonciere\" IS NULL
    GROUP BY 1, 2
    ORDER BY 3 DESC
    LIMIT 10
""").fetchall()
for typ, nat, nb, pct in r:
    print(f"  {typ:<35} {nat:<35} {nb:>6,} ({pct:>5.1f} %)".replace(",", " "))

Lignes sans valeur fonciere : 31 142 (0.67 %)

Par type de local et nature de mutation (top 10) :
  (vide)                              Vente                               21 187 ( 68.0 %)
  Dépendance                          Vente                                2 485 (  8.0 %)
  Appartement                         Vente                                2 329 (  7.5 %)
  (vide)                              Echange                              2 285 (  7.3 %)
  Local industriel. commercial ou assimilé Vente                                1 272 (  4.1 %)
  Maison                              Vente                                  934 (  3.0 %)
  (vide)                              Expropriation                          306 (  1.0 %)
  (vide)                              Vente en l'état futur d'achèvement     124 (  0.4 %)
  (vide)                              Vente terrain à bâtir                   40 (  0.1 %)
  Dépendance                          Expropriation                           

## Cellule 7 — Lignes sans valeur fonciere : test multi-lignes

Si les lignes sans prix font partie de mutations ou d'autres lignes ont un prix, cela confirme l'hypothese du mecanisme multi-lignes. Sinon, ce sont des mutations entieres sans prix.

In [7]:
r = con.execute(f"""
    WITH sans_vf AS (
        SELECT \"Date mutation\", \"Commune\", \"No disposition\"
        FROM '{pq}'
        WHERE \"Valeur fonciere\" IS NULL
    ),
    mutations AS (
        SELECT
            sv.\"Date mutation\",
            sv.\"Commune\",
            sv.\"No disposition\",
            COUNT(*) AS nb_lignes_mutation,
            COUNT(dvf.\"Valeur fonciere\") AS nb_avec_prix
        FROM sans_vf sv
        JOIN '{pq}' dvf
            ON sv.\"Date mutation\" = dvf.\"Date mutation\"
            AND sv.\"Commune\" = dvf.\"Commune\"
            AND sv.\"No disposition\" = dvf.\"No disposition\"
        GROUP BY 1, 2, 3
    )
    SELECT
        CASE
            WHEN nb_avec_prix > 0 THEN 'Mutation avec au moins une ligne avec prix'
            ELSE 'Mutation entierement sans prix'
        END AS categorie,
        COUNT(*) AS nb_mutations,
        SUM(nb_lignes_mutation) AS nb_lignes_total
    FROM mutations
    GROUP BY 1
""").fetchall()

print(f"{'Categorie':<50} {'Mutations':>12} {'Lignes':>12}")
print("-" * 76)
for cat, nb_mut, nb_lig in r:
    print(f"  {cat:<48} {nb_mut:>10,} {nb_lig:>10,}".replace(",", " "))

Categorie                                             Mutations       Lignes
----------------------------------------------------------------------------
  Mutation entierement sans prix                        5 264    953 037
  Mutation avec au moins une ligne avec prix            1 520  1 486 613


## Cellule 8 — Exemples de lignes sans prix dans leur mutation

Apercu de quelques lignes sans valeur fonciere, affichees dans le contexte de leur mutation complete, pour voir concretement le mecanisme.

In [8]:
exemples = con.execute(f"""
    SELECT DISTINCT \"Date mutation\", \"Commune\", \"No disposition\"
    FROM '{pq}'
    WHERE \"Valeur fonciere\" IS NULL
    AND \"Nature mutation\" = 'Vente'
    AND \"Type local\" IS NOT NULL
    LIMIT 3
""").fetchall()

for date, commune, disp in exemples:
    print(f"\n{'='*80}")
    print(f"Mutation : {date}, {commune}, disposition {disp}")
    print(f"{'='*80}")
    r = con.execute(f"""
        SELECT
            \"Type local\",
            \"Surface reelle bati\",
            \"Surface terrain\",
            \"Valeur fonciere\",
            \"Nature mutation\"
        FROM '{pq}'
        WHERE \"Date mutation\" = '{date}'
            AND \"Commune\" = '{commune}'
            AND \"No disposition\" = '{disp}'
        ORDER BY \"Valeur fonciere\" DESC NULLS LAST
    """).fetchall()
    print(f"  {'Type local':<30} {'Surf.batie':>10} {'Surf.terrain':>12} {'Val.fonciere':>14} {'Nature':>10}")
    print(f"  {'-'*80}")
    for typ, sb, st, vf, nat in r:
        typ_s = str(typ) if typ else '(vide)'
        sb_s = f"{sb:,}".replace(",", " ") if sb is not None else '(null)'
        st_s = f"{st:,}".replace(",", " ") if st is not None else '(null)'
        vf_s = f"{vf:,}".replace(",", " ") if vf is not None else '(null)'
        print(f"  {typ_s:<30} {sb_s:>10} {st_s:>12} {vf_s:>14} {nat:>10}")


Mutation : 2022-05-02, MOROGUES, disposition 000001
  Type local                     Surf.batie Surf.terrain   Val.fonciere     Nature
  --------------------------------------------------------------------------------
  (vide)                             (null)          245         (null)      Vente
  Maison                                 61           70         (null)      Vente
  Dépendance                              0           70         (null)      Vente
  Dépendance                              0           70         (null)      Vente
  (vide)                             (null)           40         (null)      Vente
  (vide)                             (null)           52         (null)      Vente
  (vide)                             (null)           21         (null)      Vente
  (vide)                             (null)          112         (null)      Vente
  (vide)                             (null)          186         (null)      Vente
  (vide)                          

## Cellule 9 — Synthese de l'etape 2d

In [9]:
print("Synthese — Anomalies de completude DVF 2022")
print("=" * 55)
print()

# Lots / Carrez
for lot_col, carrez_col in lots:
    r = con.execute(f"""
        SELECT
            COUNT(CASE WHEN \"{lot_col}\" IS NULL AND \"{carrez_col}\" IS NOT NULL THEN 1 END)
        FROM '{pq}'
    """).fetchone()[0]
    if r > 0:
        print(f"  {carrez_col} sans {lot_col} : {r:,} lignes".replace(",", " "))

print()
print(f"  Lignes sans Code postal   : {nb_sans_cp:,}".replace(",", " "))
print(f"  Lignes sans Valeur fonciere : {nb_sans_vf:,}".replace(",", " "))

Synthese — Anomalies de completude DVF 2022


  Lignes sans Code postal   : 38 707
  Lignes sans Valeur fonciere : 31 142


## Cellule 10 — Fermeture

In [10]:
con.close()
print("Connexion DuckDB fermee.")

Connexion DuckDB fermee.
